
# Zenith Learn — Entraînement et Validation du Modèle (Skills2Job)

Ce notebook documente l'implémentation et l'entraînement du moteur de recommandation hybride Zenith Learn, en respectant scrupuleusement les 7 Twists du Skills2Job.

**Twists couverts :**
- **T01** : Score hybride (Engagement + Complétion)
- **T03** : Neutralisation du biais de durée
- **T04** : Intégrité des signaux (Action vs Business)
- **T05** : Segmentation des profils (Actif vs Explorateur)
- **T06** : Explicabilité déterministe
- **T07** : Classement du potentiel de réussite


In [ ]:

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ajouter le parent au path pour importer nos modules
sys.path.append(os.path.abspath('..'))

from data.loader import load_course_metadata, load_learner_ratings
from data.preprocessor import preprocess_interactions
from model.recommender import ZenithRecommender
from model.explainer import RecommendationExplainer

# Configuration visuelle
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]



## 1. Chargement et Exploration des Données


In [ ]:

courses_df = load_course_metadata()
ratings_df = load_learner_ratings()

print(f"Nombre de cours : {len(courses_df)}")
print(f"Nombre d'interactions : {len(ratings_df)}")
print(f"Nombre d'apprenants : {ratings_df['learner_id'].nunique()}")

# Distribution des notes brutes
sns.histplot(ratings_df['rating'], bins=5, kde=True)
plt.title("Distribution des Notes Brutes")
plt.show()



## 2. Préprocessing et Ingénierie des Caractéristiques (T01, T03, T04)
Ici, nous appliquons :
- **T01** : Calcul du score d'engagement.
- **T03** : Normalisation logarithmique de la complétion pour éviter le biais des cours courts.


In [ ]:

interactions_df = preprocess_interactions(ratings_df, courses_df)

# Visualisation du biais de durée corrigé (T03)
sns.scatterplot(data=interactions_df, x='duration_hours', y='adjusted_completion', alpha=0.3)
plt.title("Complétion Ajustée vs Durée du Cours (T03)")
plt.show()



## 3. Entraînement du Modèle Hybride (T01, T05)
On entraîne le recommandeur et on observe la segmentation automatique.


In [ ]:

recommender = ZenithRecommender(alpha_engagement=0.55, beta_completion=0.45)
recommender.fit(interactions_df, courses_df)

# Stats de segmentation (T05)
segments = pd.Series(recommender.user_segments).value_counts()
segments.plot(kind='pie', autopct='%1.1f%%', colors=['#4CAF50', '#2196F3'])
plt.title("Répartition des Segments (T05)")
plt.show()



## 4. Validation de l'Explicabilité (T06)


In [ ]:

explainer = RecommendationExplainer(recommender)
sample_uid = interactions_df['learner_id'].iloc[0]
sample_cid = interactions_df['course_id'].iloc[0]

explanation = explainer.explain_recommendation_for_user(sample_uid, sample_cid)
print(explanation['explanation_text'])



## 5. Classement du Top 100 Success Potential (T07)


In [ ]:

top_learners = recommender.rank_learners_by_success_potential(top_n=10)
top_df = pd.DataFrame(top_learners)
top_df['success_score'] = top_df['score']
print("Top 10 Apprenants (Potentiel de Réussite) :")
display(top_df[['learner_id', 'success_score', 'metrics']])
